# Phase 4: QLoRA Fine-Tuning (Colab Environment)

This notebook is intended to be run in Google Colab with a T4 GPU, as `bitsandbytes` 4-bit quantization may not be supported on macOS or without an NVIDIA GPU.

In [ ]:
!pip install -q transformers peft datasets accelerate bitsandbytes mlflow scikit-learn

In [ ]:
import torch
from transformers import (
    AutoModelForSequenceClassification, 
    AutoTokenizer, 
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)
from peft import get_peft_model, LoraConfig, TaskType
from datasets import load_dataset
import mlflow
import numpy as np
from sklearn.metrics import f1_score

set_seed(42)

In [ ]:
model_name = "xlm-roberta-base"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=4, 
    quantization_config=bnb_config,
    device_map="auto"
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "value"]
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Load datasets (make sure you upload or mount them to Colab)
dataset = load_dataset("json", data_files={"train": "data/processed/semeval_train.jsonl"})

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"macro_f1": f1_score(labels, predictions, average="macro")}

In [ ]:
training_args = TrainingArguments(
    output_dir="./models/sentiment/qlora-adapter",
    evaluation_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()
trainer.model.save_pretrained("./models/sentiment/qlora-adapter")